# Phase 3 — Model H (Hindi): reasoning finetuning and analysis

Finetunes the Phase 2 Hindi checkpoint on a synthetic comparative-reasoning
corpus, evaluates it against the pretrained model, and compares attention
before and after.

**What this notebook establishes, in order.** The corpus is generated here
rather than downloaded, so the leakage checks run on the exact bytes that get
trained on. Two finetunes follow — one supervised on bare answers, one on a
restated inequality chain — because which supervision a 25M model can actually
use is an empirical question, not one to guess. The pretrained model is scored
both zero-shot and few-shot: zero-shot shows it has no notion of the answer
format, few-shot supplies the format so the remaining gap is about reasoning.

Total GPU time is around **35 minutes**, most of it evaluation rather than
training. Every cell is resumable; if Colab disconnects mid-finetune, re-run
the training cell and it continues from `finetune_latest.pt`.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

In [ ]:
import shutil, os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/24KaratAu'
WORK = '/content/phase3_hindi'

if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(f'{DRIVE_BASE}/code', WORK)
%cd {WORK}

In [ ]:
!apt-get install -y -qq fonts-indic 2>&1 | tail -2
!pip install -q pyyaml sentencepiece

## 2. Build the reasoning corpus

Generated from `src/reasoning/generate.py` with a fixed seed, so it is
reproducible byte-for-byte. The generator refuses to finish if any entity or
any prompt is shared between train and test.

In [ ]:
!python -u -m src.reasoning.generate --language hindi \
    --out-dir data/hindi/reasoning \
    --train 40000 --val 5000 --test 5000 --seed 42

### Verify the corpus before spending GPU time

Two things are checked: that the generator's own leakage report is clean, and
that the labels are actually right. The second is the one that matters — a
premise renderer that states an inequality backwards would produce a corpus
that trains and evaluates without complaint while being silently wrong. The
test suite re-parses the rendered Devanagari back into an ordering and checks
every answer against it.

In [ ]:
import json
stats = json.load(open('data/hindi/reasoning/stats.json'))
print(json.dumps(stats['leakage'], ensure_ascii=False, indent=2))
assert not stats['leakage']['train_test_shared_entities']
assert stats['leakage']['train_test_shared_prompts'] == 0
assert stats['leakage']['val_test_shared_prompts'] == 0

for split in ('train', 'val', 'test'):
    s = stats['splits'][split]
    print(f"{split:<6} n={s['n']:>6,}  families={len(s['families'])}  "
          f"hops={s['hops']}  held-out phrasing={s['held_out_phrasing']:,}")
print('\ntokenizer:', json.dumps(stats['tokenizer']['train'], ensure_ascii=False))

In [ ]:
!python -m pytest tests/ -q

In [ ]:
# Archive the corpus to Drive: 46 MB is too much for git, and the brief wants
# large artifacts on Drive with links in the README.
!mkdir -p {DRIVE_BASE}/hindi/reasoning
!cp data/hindi/reasoning/*.jsonl data/hindi/reasoning/stats.json \
    data/hindi/reasoning/sample.md {DRIVE_BASE}/hindi/reasoning/
!ls -la {DRIVE_BASE}/hindi/reasoning

## 3. Finetune (~6 min per run)

Both runs start from the *same* pretrained checkpoint and differ only in what
the target span contains. Weights are loaded without the optimizer state: the
pretraining run ended at a 3e-5 learning rate, and restoring AdamW's second
moments from that point would damp exactly the updates this phase depends on.

In [ ]:
import yaml
cfg = yaml.safe_load(open('hindi/configs/finetune.yaml'))
assert cfg['answer_style'] == 'direct'
print('init from :', cfg['init_from'])
print('checkpoints:', cfg['checkpoint']['dir'])
assert os.path.exists(cfg['init_from']), 'Phase 2 checkpoint not found on Drive'

In [ ]:
!python -u -m src.train.finetune --config hindi/configs/finetune.yaml

In [ ]:
!python -u -m src.train.finetune --config hindi/configs/finetune_rationale.yaml

## 4. Pretrained vs finetuned exact match

Three setups per answer style. The zero-shot number is expected to be near
zero and is reported anyway: the point is that the gap it shows is a formatting
gap, which the few-shot run then removes.

In [ ]:
FT_D = f'{DRIVE_BASE}/hindi/checkpoints_ft_direct/finetune_best.pt'
FT_R = f'{DRIVE_BASE}/hindi/checkpoints_ft_rationale/finetune_best.pt'
PRE  = f'{DRIVE_BASE}/hindi/checkpoints/checkpoint_best.pt'
COMMON = ('--model-config hindi/configs/model.yaml '
          '--tokenizer tokenizers/hindi/hindi_bpe_8k.model '
          '--test data/hindi/reasoning/test.jsonl --language hindi')
os.makedirs('hindi/eval', exist_ok=True)

In [ ]:
!python -u -m src.eval.reasoning_eval --checkpoint {PRE} {COMMON} \
    --mode zeroshot --answer-style direct \
    --out hindi/eval/reasoning_zeroshot_direct.json

!python -u -m src.eval.reasoning_eval --checkpoint {PRE} {COMMON} \
    --mode fewshot --shots 4 --train data/hindi/reasoning/train.jsonl \
    --answer-style direct --out hindi/eval/reasoning_fewshot_direct.json

In [ ]:
!python -u -m src.eval.reasoning_eval --checkpoint {FT_D} {COMMON} \
    --mode finetuned --answer-style direct \
    --out hindi/eval/reasoning_finetuned_direct.json

!python -u -m src.eval.reasoning_eval --checkpoint {FT_R} {COMMON} \
    --mode finetuned --answer-style rationale \
    --out hindi/eval/reasoning_finetuned_rationale.json

## 5. Attention: pretrained vs finetuned

Both checkpoints are run over the *same* reasoning prompts, batched without
padding so that learned absolute positions are not shifted. Heatmaps cover one
early layer (0) and one late layer (7), as the brief requires.

In [ ]:
!python -u -m src.eval.attention_reasoning \
    --pretrained {PRE} --finetuned {FT_D} \
    --model-config hindi/configs/model.yaml \
    --tokenizer tokenizers/hindi/hindi_bpe_8k.model \
    --test data/hindi/reasoning/test.jsonl --language hindi --tag direct \
    --outdir report/figures --out hindi/eval/attention_reasoning.json

## 6. Copy results back to Drive

In [ ]:
!mkdir -p {DRIVE_BASE}/hindi/phase3
!cp -r hindi/eval {DRIVE_BASE}/hindi/phase3/
!cp -r report/figures {DRIVE_BASE}/hindi/phase3/
!cp {DRIVE_BASE}/hindi/logs/finetune_*.jsonl hindi/logs/ 2>/dev/null || true
!ls hindi/eval report/figures | head -40
print('\nCommit from your machine: hindi/eval/*.json, hindi/logs/finetune_*.jsonl,')
print('report/figures/*, data/hindi/reasoning/{stats.json,sample.md}')